<div style="text-align: justify; max-width: 90ch">

# Ollama setup — a free local LLM for the GenAI track

*Part of `setup/`. Do this once, before the `gen_ai/` track. No MLflow knowledge needed.*

By the end of this notebook you will have:

- Ollama installed and running in the background.
- The track's default model, `gemma3:4b`, downloaded and answering, plus the two extra models that single notebooks need:
  `qwen3:1.7b` for tool calling and `nomic-embed-text` for embeddings.
- A working Python call through the same endpoint every GenAI notebook uses, including a tool call.
- Fixes for the problems readers hit most often.

This is a standalone setup guide with no upstream MLflow tutorial behind it. Install commands follow Ollama's official documentation at [docs.ollama.com](https://docs.ollama.com).
They change occasionally, so check there if one fails.

</div>

<div style="text-align: justify; max-width: 90ch">

## Why a local model?

The `gen_ai/` notebooks need a **large language model (LLM)** to trace, evaluate, and serve. The obvious choice, a hosted API such as OpenAI's, brings three problems for a course:

- **Cost.** Every call is billed. Re-running a notebook, or a class re-running it, adds up.
- **An API key.** You must sign up, add a payment method, and keep the key out of Git.
- **Privacy.** Every prompt leaves your machine, which rules out many research datasets.

A **local model** runs on your own computer: no cost per call, no key, and nothing leaves the machine.

The price is quality and speed. A model small enough for a laptop is far weaker than a hosted one. That is fine for learning MLflow, where the point is to *see* and *measure* the calls, not to get brilliant answers.
Every notebook in the track runs on local models, including the LLM judge that grades answers. Where a stronger hosted model would help, the notebook shows the swap as an optional step.

</div>

<div style="text-align: justify; max-width: 90ch">

## What Ollama is

**Ollama** is a free, open-source program that downloads LLMs and runs them on your machine behind a small web server. Your code talks to that server over HTTP,
the same way a notebook talks to the MLflow tracking server in `basics/a_setup_mlflow`.

```text
your notebook (openai client) --HTTP--> Ollama server at localhost:11434 --> model file on disk, run on GPU or CPU
```

| Term                      | Meaning                                                                                                                                                  |
| ------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **service** (or *daemon*) | A program the operating system keeps running in the background. Ollama's installers set one up, so unlike the MLflow server you rarely start it by hand. |
| **model weights**         | The model's learned numbers, stored as one large file (in the GGUF format) that Ollama loads into memory.                                                |
| **`localhost:11434`**     | Your own machine (`localhost`), port `11434`: Ollama's default address.                                                                                  |
| **OpenAI-compatible API** | Ollama accepts the same requests as OpenAI's API, at `http://localhost:11434/v1`. Code written for OpenAI works after changing one URL.                  |

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 1: Install Ollama

Pick your operating system. Each installer puts the `ollama` command on your `PATH` and starts the server in the background.

**GPU drivers.** Ollama uses NVIDIA GPUs with driver 550 or newer (551.61 or newer on Windows), AMD GPUs with ROCm v7 drivers,
and Apple Silicon GPUs. Without a supported GPU it falls back to the CPU automatically. There is nothing extra to install for that.

### Linux

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

The script asks for your `sudo` password. On a distribution that uses systemd (most do), it:

- installs the `ollama` binary,
- creates a system user called `ollama`,
- registers an `ollama` systemd service, then enables and starts it.

Prefer not to pipe a script into your shell? Read it first at <https://ollama.com/install.sh>, or follow the [manual install](https://docs.ollama.com/linux).
On Windows Subsystem for Linux the script needs WSL2; WSL1 is not supported.

### macOS

Requires macOS 14 Sonoma or newer. Apple Silicon (M-series) Macs run models on the GPU; Intel Macs run them on the CPU only.

1. Download `Ollama.dmg` from <https://ollama.com/download/mac>.
2. Open it and drag **Ollama** into **Applications**.
3. Launch Ollama. If the `ollama` command isn't on your `PATH` yet, it asks permission to create a link in `/usr/local/bin`.
   Accept.

The app lives in the menu bar, runs the server, and registers itself to start at login. The same download page also offers a terminal install,
`curl -fsSL https://ollama.com/install.sh | sh`, which installs the app into `/Applications` for you.

### Windows

Requires Windows 10 22H2 or newer. No administrator rights needed.

- **Installer:** download and run `OllamaSetup.exe` from <https://ollama.com/download/windows>.
- **Or PowerShell:**

  ```powershell
  irm https://ollama.com/install.ps1 | iex
  ```

Ollama installs into your user profile, adds `ollama` to your `PATH`, runs in the system tray, and starts at login. Open a **new** terminal afterwards so it picks up the updated `PATH`.

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 2: Check that the server is running

In a terminal:

```bash
ollama --version
curl http://localhost:11434        # prints: Ollama is running
```

In Windows PowerShell, type `curl.exe` instead of `curl`: there, `curl` can be an alias for a different command.

If the second command can't connect, start the server:

| OS                          | Start it                                    | Status and logs                                         |
| --------------------------- | ------------------------------------------- | ------------------------------------------------------- |
| Linux                       | `sudo systemctl start ollama`               | `systemctl status ollama` and `journalctl -e -u ollama` |
| macOS                       | Open the Ollama app                         | `cat ~/.ollama/logs/server.log`                         |
| Windows                     | Open Ollama from the Start menu             | `server.log` in `%LOCALAPPDATA%\Ollama`                 |
| Any OS, without the service | `ollama serve` in a terminal you leave open | printed in that terminal                                |

Now the same check from Python, the path every notebook takes. This cell uses Ollama's **native API** (paths under `/api/`) for housekeeping.
Chat comes later, through the OpenAI-compatible API. `urllib` is part of the standard library, so nothing needs installing.

</div>

In [1]:
import json
import urllib.error
import urllib.request

OLLAMA_URL = "http://localhost:11434"


def get(path: str):
    """GET a path on the Ollama server, with a hint if the server is unreachable."""
    try:
        with urllib.request.urlopen(OLLAMA_URL + path, timeout=5) as resp:
            body = resp.read().decode()
    except urllib.error.URLError as err:
        raise RuntimeError(
            f"Can't reach Ollama at {OLLAMA_URL} ({err.reason}). "
            "Start the server (Step 2) and re-run this cell."
        ) from None
    return json.loads(body) if body.startswith("{") else body


print(get("/"))
print("Server version:", get("/api/version")["version"])

Ollama is running
Server version: 0.13.5


<div style="text-align: justify; max-width: 90ch">

## Choosing a model: tags, sizes and quantization

Decide which model fits your machine before downloading one. The model has to fit in memory; everything else follows from that.

### Reading a model tag

Ollama names models `name:tag`, for example `gemma3:4b`.

| Part                | Example                              | Meaning                                                                                                                                                    |
| ------------------- | ------------------------------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------------- |
| name                | `gemma3`                             | The model family: Gemma 3, from Google DeepMind.                                                                                                           |
| size                | `4b`                                 | Size label in billions of **parameters** (learned weights). More parameters: better answers, more memory, slower.                                          |
| quantization suffix | `-it-q4_K_M`, `-it-q8_0`, `-it-fp16` | How many bits store each weight (next section). `it` means *instruction-tuned*, the chat variant. For `gemma3:4b`, leaving it off gives `q4_K_M`.          |
| no tag at all       | `gemma3`                             | Means `gemma3:latest`. Here that is the 4b model, but `latest` can be a much bigger model: `qwen3` alone is `qwen3:8b` (5.2 GB). Always type the full tag. |

### What quantization trades away

Model weights are normally stored as 16-bit numbers. **Quantization** stores them with fewer bits, so the file shrinks and fits into less memory.

**`Q4_K_M`** is a popular middle ground, averaging roughly 4 to 5 bits per weight:

- **`Q4`**: 4-bit blocks for most weights.
- **`K`**: llama.cpp's "k-quant" scheme, which keeps some sensitive tensors at higher precision.
- **`M`**: the medium variant (there are also `S`, small, and `L`, large).

The same model at three precisions (download sizes from the [Ollama library](https://ollama.com/library/gemma3/tags)):

| Tag                                         | Bits per weight | Download |
| ------------------------------------------- | --------------- | -------- |
| `gemma3:4b-it-fp16`                         | 16              | 8.6 GB   |
| `gemma3:4b-it-q8_0`                         | 8               | 5.0 GB   |
| `gemma3:4b` (same as `gemma3:4b-it-q4_K_M`) | about 4 to 5    | 3.3 GB   |

What `Q4_K_M` costs you:

- **A little accuracy.** Answers stay close to full precision, but slip slightly more often on exact tasks: arithmetic, strict JSON, a tool call's arguments.
- **More of it on small models.** A 4B model has less spare capacity to absorb rounding error than a 12B one.

What you get: less than half the memory of `fp16`, and faster generation, because shuttling weights through memory is usually the bottleneck.
A common rule of thumb: for the same memory, a bigger model at 4 bits beats a smaller model at full precision.

### Picking a size for your machine

Where a model runs decides how fast it answers:

- **GPU memory (VRAM):** fastest.
- **System RAM, on the CPU:** the automatic fallback. It works, just several times slower.
- **Split between the two:** when the model only partly fits in VRAM. Usually much slower than all-GPU.
- **Apple Silicon:** CPU and GPU share one pool of memory. Count your total RAM, minus what macOS and your open apps use.

Plan for the download size plus headroom for the **context window**, the model's working memory for the conversation (covered under settings below).
A loaded model takes more memory than its download; Step 6 shows the actual number on your machine.

A rough guide; leave 1 to 2 GB spare:

| Your machine                                                                  | Start with                     | Download |
| ----------------------------------------------------------------------------- | ------------------------------ | -------- |
| CPU only (8 GB RAM or more), a GPU with 6 GB VRAM or more, or a Mac with 8 GB | `gemma3:4b`, the track default | 3.3 GB   |
| Very small machines                                                           | `gemma3:1b`                    | 0.8 GB   |
| GPU with 12 GB VRAM or more, or a Mac with 16–24 GB                           | `gemma3:12b`                   | 8.1 GB   |
| GPU with 24 GB VRAM or more, or a Mac with 32 GB or more                      | `gemma3:27b`                   | 17 GB    |

**CPU-only machines:** a bigger model may *fit* in RAM and still be too slow to be pleasant. Stay with `gemma3:4b` unless you're patient.
`gemma3:1b` answers noticeably worse, and the notebooks that use the model as a judge become unreliable with it.

### Why the track defaults to `gemma3:4b`

Every GenAI notebook uses `gemma3:4b` unless you change it. The choice balances three needs:

- **Small enough for most machines.** It fits on a 6 GB GPU with room for its context window, and stays usable on a CPU.
- **It answers directly.** Gemma 3 is not a *reasoning* model: it does not write a hidden thinking trace before the answer (Step 5 shows one that does).
  That keeps traces short and fast, and its replies parse cleanly where the track needs structured output: the LLM judge in `c_`, `e_`, `g_` and `i_`, and DSPy in `h_`.
- **Good enough to grade with.** For the clear-cut checks the notebooks make (is the answer relevant? is it one sentence?) it works as a local judge, so the whole track stays free.

The trade-off: it is a small model. It states wrong facts confidently and its reasoning is shallow. That's acceptable for learning MLflow.
If you have the memory, `gemma3:12b` gives better answers with no other changes: each notebook sets the model name in one place, a `MODEL` variable or a `model=` argument.

Two notebooks need something `gemma3:4b` cannot do, so they use a second model:

- **`gen_ai/d_langchain_agent`** builds an agent that calls Python functions. `gemma3:4b` has no tool-calling support (Step 3 shows how to check),
  so that notebook uses **`qwen3:1.7b`** (1.4 GB), the smallest Qwen3 that still calls tools reliably: in a five-prompt tool-calling test through Ollama's OpenAI-compatible endpoint it made the correct call 5 times out of 5,
  while `qwen3:0.6b` missed 2 (Step 7).
- **`gen_ai/i_rag_capstone`** needs an **embedding model**, which turns text into vectors for retrieval; a chat model cannot do that.
  It uses **`nomic-embed-text`** (0.3 GB). The capstone explains what an embedding is.

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 3: Download and manage models

A model is either **downloaded** (a file on disk) or also **loaded** (in memory, ready to answer).

| Command                        | What it does                                                          |
| ------------------------------ | --------------------------------------------------------------------- |
| `ollama pull gemma3:4b`        | Download the model.                                                   |
| `ollama list` (or `ollama ls`) | Show downloaded models and their sizes.                               |
| `ollama show gemma3:4b`        | Show details: parameters, context length, quantization, capabilities. |
| `ollama run gemma3:4b`         | Load the model if needed, then chat in the terminal.                  |
| `ollama ps`                    | Show loaded models, and whether they sit on the GPU or CPU (Step 6).  |
| `ollama stop gemma3:4b`        | Unload it from memory now. The download stays.                        |
| `ollama rm gemma3:4b`          | Delete the download.                                                  |

Download the track's models and confirm they're there. Only `gemma3:4b` is needed to start; the other two can wait until you reach the notebook that uses them:

```bash
ollama pull gemma3:4b         # the default: generation and the LLM judge, every notebook
ollama pull qwen3:1.7b        # gen_ai/d_langchain_agent only: tool calling
ollama pull nomic-embed-text  # gen_ai/i_rag_capstone only: embeddings
ollama list
```

```text
NAME                       ID              SIZE      MODIFIED
nomic-embed-text:latest    0a109f422b47    274 MB    2 minutes ago
qwen3:1.7b                 8f68893c685c    1.4 GB    5 minutes ago
gemma3:4b                  a2af6cc3eb7f    3.3 GB    12 minutes ago
```

Check what the default model can do:

```bash
ollama show gemma3:4b
```

```text
  Model
    architecture        gemma3
    parameters          4.3B
    context length      131072
    embedding length    2560
    quantization        Q4_K_M

  Capabilities
    completion
    vision
```

The **Capabilities** list is the line to read before choosing any model for the track. `gemma3:4b` offers `completion` (chat) and `vision` (it also accepts images, unused here).
Two capabilities it lacks explain the second chat model:

- **`tools`**: the model supports tool calling (Step 7). `ollama show qwen3:1.7b` lists it; `gemma3:4b` does not, which is why `d_langchain_agent` uses `qwen3:1.7b`.
- **`thinking`**: the model can write a reasoning trace before answering (Step 5). Again `qwen3:1.7b` has it and `gemma3:4b` does not.

`context length 131072` is the most the model *supports*. Ollama allocates less by default (settings, below).

Try a chat in the terminal:

```bash
ollama run gemma3:4b                             # interactive: /? lists commands, /bye leaves
ollama run gemma3:4b "Say hi in three words."    # one-shot
```

Add `--verbose` to a one-shot run to print timings, including `load duration`: how long loading into memory took. Ollama unloads an idle model after 5 minutes.
`ollama stop gemma3:4b` frees the memory straight away.

The same "is it downloaded?" check from Python, for the three models the track uses:

</div>

In [2]:
MODEL = "gemma3:4b"  # the track default; change it if you pulled a different chat model
TOOL_MODEL = (
    "qwen3:1.7b"  # gen_ai/d_langchain_agent only: it needs tool calling (Step 7)
)
EMBED_MODEL = (
    "nomic-embed-text:latest"  # gen_ai/i_rag_capstone only: an embedding model
)

downloaded = {m["name"]: m for m in get("/api/tags")["models"]}
for name in (MODEL, TOOL_MODEL, EMBED_MODEL):
    info = downloaded.get(name)
    if info is None:
        print(f"{name:<24} not downloaded yet: run `ollama pull {name}`")
    else:
        print(
            f"{name:<24} {info['size'] / 1e9:4.1f} GB  {info['details']['quantization_level']}"
        )

if MODEL not in downloaded:
    raise RuntimeError(
        f"{MODEL} is not downloaded yet. In a terminal, run: ollama pull {MODEL}"
    )

gemma3:4b                 3.3 GB  Q4_K_M
qwen3:1.7b                1.4 GB  Q4_K_M
nomic-embed-text:latest   0.3 GB  F16


<div style="text-align: justify; max-width: 90ch">

## Where models live, and the settings you can change

**For this course the defaults are fine.** Skip this section unless you're short on disk space or a later step tells you to change something.

### Where downloads go

| OS                        | Default model folder                 |
| ------------------------- | ------------------------------------ |
| Linux (installed service) | `/usr/share/ollama/.ollama/models`   |
| macOS                     | `~/.ollama/models`                   |
| Windows                   | `C:\Users\%username%\.ollama\models` |

Models are large, so a small system disk fills up fast. `OLLAMA_MODELS` (below) moves them to a bigger drive.

### Environment variables

The Ollama server reads its settings from environment variables. The ones worth knowing:

| Variable                | Default                                                                 | What it changes                                                                                                                                                                                           |
| ----------------------- | ----------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `OLLAMA_HOST`           | `127.0.0.1:11434`                                                       | The address the server listens on. The `ollama` command reads it too, to find the server. `0.0.0.0:11434` exposes Ollama to your network, with no authentication, so only do that on a network you trust. |
| `OLLAMA_MODELS`         | the folders above                                                       | Where models are stored.                                                                                                                                                                                  |
| `OLLAMA_CONTEXT_LENGTH` | set by your VRAM: 4k tokens below 24 GiB, 32k for 24–48 GiB, 256k above | The context window allocated to each loaded model.                                                                                                                                                        |
| `OLLAMA_KEEP_ALIVE`     | `5m`                                                                    | How long an idle model stays loaded.                                                                                                                                                                      |
| `CUDA_VISIBLE_DEVICES`  | all NVIDIA GPUs                                                         | `-1` forces CPU-only, handy for comparing speeds.                                                                                                                                                         |

**Context window.** The number of **tokens** (word pieces) the model can see at once: prompt plus answer. `gemma3:4b` supports up to 131,072, but memory use grows with the window, so Ollama allocates less.
Short prompts, like the ones in the GenAI notebooks, fit comfortably in 4k. Raise it only for long documents, then check `ollama ps` still shows the model on the GPU.

### The gotcha: settings must reach the server

`export OLLAMA_CONTEXT_LENGTH=8192` in your terminal changes nothing for a server that is already running as a service. Set the variable where the server reads it, then restart the server:

- **Linux (systemd):** open an override file for the service:

  ```bash
  sudo systemctl edit ollama.service
  ```

add the variable under `[Service]`:

  ```ini
  [Service]
  Environment="OLLAMA_CONTEXT_LENGTH=8192"
  ```

then reload and restart:

  ```bash
  sudo systemctl daemon-reload
  sudo systemctl restart ollama
  ```

If you move `OLLAMA_MODELS`, the service's `ollama` user needs read and write access: `sudo chown -R ollama:ollama <directory>`.

- **macOS:** run `launchctl setenv OLLAMA_CONTEXT_LENGTH 8192`, then quit and reopen the Ollama app.
- **Windows:** quit Ollama from the system tray. Open **Settings**, search for "environment variables", choose **Edit environment variables for your account**,
  add the variable, then start Ollama again.
- **A one-off server in a terminal:** `OLLAMA_CONTEXT_LENGTH=8192 ollama serve`. Stop the service or app first, or the port is taken (Troubleshooting).

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 4: Call the model from Python

The same server offers two APIs:

| API                   | Address                          | Used for                                                                                                        |
| --------------------- | -------------------------------- | --------------------------------------------------------------------------------------------------------------- |
| **Native**            | `http://localhost:11434/api/...` | Ollama-specific housekeeping, like the cells above: what's downloaded (`/api/tags`), what's loaded (`/api/ps`). |
| **OpenAI-compatible** | `http://localhost:11434/v1`      | Chat and tool calls in the request format of OpenAI's API. **Every GenAI notebook uses this one.**              |

Why the compatible one? The `openai` Python client, LangChain's `ChatOpenAI`, MLflow's `mlflow.openai.autolog()` and many other tools already speak OpenAI's format.
Pointing them at Ollama takes two settings:

| Setting    | Value                            | Why                                                         |
| ---------- | -------------------------------- | ----------------------------------------------------------- |
| `base_url` | `http://localhost:11434/v1`      | Send requests to Ollama instead of OpenAI's servers.        |
| `api_key`  | any placeholder, e.g. `"ollama"` | The client refuses to start without one. Ollama ignores it. |

The `openai` client is already a dependency of this repo, so `uv sync` installed it.

</div>

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by the client, ignored by Ollama
    timeout=120,  # generous: the first call may wait for the model to load
    max_retries=0,  # fail fast instead of retrying silently
)

# The endpoint lists every model you've pulled; keep the ones the track uses.
print(
    sorted(
        m.id for m in client.models.list() if m.id in (MODEL, TOOL_MODEL, EMBED_MODEL)
    )
)

['gemma3:4b', 'nomic-embed-text:latest', 'qwen3:1.7b']


<div style="text-align: justify; max-width: 90ch">

### A minimal chat call

This is the call shape you'll meet in `gen_ai/a_tracing_quickstart`: a model name and a list of messages. The cell also times the call and counts the tokens the model generated.

</div>

In [4]:
import time

QUESTION = "Which planet is closest to the Sun? Reply with one word."

start = time.perf_counter()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)
elapsed = time.perf_counter() - start

answer = response.choices[0].message
print("Answer:", answer.content)
print(f"Took {elapsed:.1f} s and generated {response.usage.completion_tokens} tokens")

Answer: Mercury
Took 2.1 s and generated 2 tokens


<div style="text-align: justify; max-width: 90ch">

Timings depend entirely on your hardware, so read the numbers stored here as one example. If the model wasn't loaded yet, this call also paid for loading it, and the next one will be quicker.

The token count is about what a one-word answer needs: `gemma3:4b` answers directly. The next step shows a model that does not.

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 5: Qwen3's thinking mode

`gen_ai/d_langchain_agent` uses `qwen3:1.7b`, a **reasoning model**. Before answering, it writes a **thinking trace**, a step-by-step monologue, and only then the answer.
Ask it the same question:

</div>

In [5]:
start = time.perf_counter()
response = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)
elapsed = time.perf_counter() - start

answer = response.choices[0].message
print("Answer:", answer.content)
print(f"Took {elapsed:.1f} s and generated {response.usage.completion_tokens} tokens")

Answer: Mercury
Took 1.4 s and generated 145 tokens


<div style="text-align: justify; max-width: 90ch">

Far more tokens than a one-word answer needs. Where did they go? Ollama keeps the two parts of the reply apart:

- **`message.content`** holds the answer.
- **`message.reasoning`** holds the thinking trace. It is an extra field the `openai` client has no named attribute for, so read it with `getattr`.

</div>

In [6]:
reasoning = getattr(answer, "reasoning", None) or ""
print(f"Thinking trace: {len(reasoning)} characters. It begins:\n")
print(reasoning[:300] + " ...")

Thinking trace: 630 characters. It begins:

Okay, the user is asking which planet is closest to the Sun and wants the answer in one word. Let me think.

I remember that the planets orbit the Sun in order from closest to farthest. The inner planets are closer. So the order is Mercury, Venus, Earth, Mars. Then the outer ones: Jupiter, Saturn, U ...


<div style="text-align: justify; max-width: 90ch">

Thinking helps with hard multi-step problems. For short calls like this one it mostly costs:

- **Time.** Every thinking token is generated like any other token.
- **Your `max_tokens` budget.** Thinking tokens count against it. Set `max_tokens` low and the model can spend it all thinking:
  you get an **empty** answer and `finish_reason="length"`.

To switch thinking off for a call, pass `reasoning_effort="none"`:

</div>

In [7]:
start = time.perf_counter()
response = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": QUESTION}],
    reasoning_effort="none",  # Ollama skips the thinking trace for this call
)
elapsed = time.perf_counter() - start

answer = response.choices[0].message
print("Answer:", answer.content)
print("Thinking trace:", getattr(answer, "reasoning", None))
print(f"Took {elapsed:.1f} s and generated {response.usage.completion_tokens} tokens")

Answer: Mercury.
Thinking trace: None
Took 0.1 s and generated 4 tokens


<div style="text-align: justify; max-width: 90ch">

The same switch in the other places you'll use Ollama:

| Where                                                 | Thinking off                                                                                  |
| ----------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| `openai` client (above)                               | `reasoning_effort="none"`                                                                     |
| LangChain's `ChatOpenAI` (`gen_ai/d_langchain_agent`) | `ChatOpenAI(..., reasoning_effort="none")`                                                    |
| Ollama's native API (`/api/chat`)                     | `"think": false` in the request body                                                          |
| Terminal, one-shot                                    | `ollama run qwen3:1.7b --think=false "..."` (`--hidethinking` keeps thinking on but hides it) |
| Terminal, interactive session                         | `/set nothink`, and `/set think` to turn it back on                                           |

`gemma3:4b` has no thinking mode, so none of this applies to the track's default. `d_langchain_agent` keeps thinking *on* deliberately:
its agent has to plan, and the notebook explains the trade-off.

**What about `/no_think`?** Qwen3 was trained to respond to `/think` and `/no_think` typed at the end of a message, and many examples online rely on it.
With current Ollama releases it is not a reliable off switch: in test runs through the OpenAI-compatible endpoint, `qwen3:1.7b` still wrote a full thinking trace with `/no_think` appended,
and only the final answer got shorter. Prefer the explicit settings in the table.

</div>

<div style="text-align: justify; max-width: 90ch">

## Step 6: GPU or CPU? Reading `ollama ps`

Both models have just answered, so they should be loaded; on a small GPU, loading the second may have evicted the first.
`ollama ps` shows what is loaded and where it sits:

```bash
ollama ps
```

In one example run on a laptop GPU, it printed:

```text
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL
qwen3:1.7b    8f68893c685c    1.9 GB    100% GPU     4096       4 minutes from now
gemma3:4b     a2af6cc3eb7f    4.3 GB    100% GPU     4096       3 minutes from now
```

| Column      | Meaning                                                                                                                                                      |
| ----------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `SIZE`      | Memory the loaded model uses: weights plus context window. Larger than the download.                                                                         |
| `PROCESSOR` | `100% GPU`: entirely in GPU memory, fastest. `100% CPU`: entirely in system RAM. `48%/52% CPU/GPU`: split between the two, usually much slower than all-GPU. |
| `CONTEXT`   | The context window allocated for this model.                                                                                                                 |
| `UNTIL`     | When Ollama unloads it if it stays idle.                                                                                                                     |

What to do with it:

- **`100% CPU` but you have a GPU:** Ollama didn't find a usable GPU. See Troubleshooting.
- **A CPU/GPU split:** the model doesn't quite fit. Pick a smaller model or a shorter context window.
- **Several models listed:** each holds its own memory until it unloads. `ollama stop <name>` frees one.

The same information from Python, through the native API. The cell first sends the default model a one-token request, so it is loaded again if it was evicted:

</div>

In [8]:
# Make sure the default model is loaded: an idle model unloads after a few minutes.
client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=1
)

loaded = {m["name"]: m for m in get("/api/ps")["models"]}
for name in (MODEL, TOOL_MODEL):
    m = loaded.get(name)
    if m is None:
        print(f"{name:<24} not loaded (idle too long, or evicted to make room)")
        continue
    on_gpu = m["size_vram"] / m["size"]
    print(
        f"{name:<24} {m['size'] / 1e9:4.1f} GB loaded, {on_gpu:.0%} on GPU, "
        f"context {m['context_length']} tokens"
    )

gemma3:4b                 4.3 GB loaded, 100% on GPU, context 4096 tokens
qwen3:1.7b                1.9 GB loaded, 100% on GPU, context 4096 tokens


<div style="text-align: justify; max-width: 90ch">

## Step 7: A tool call

An **agent** is an LLM that can act, not just talk: it decides to call functions you give it (a search, a calculator, a database query) and uses the results.
The mechanism is **tool calling**, also called *function calling*, and `gen_ai/d_langchain_agent` depends on it. `gemma3:4b` cannot do it (no `tools` in its `ollama show` output), so this step uses `TOOL_MODEL`, `qwen3:1.7b`.

The model never runs your code. One round goes like this:

1. You send the question plus a description of each tool: its name, what it does, and its arguments as a JSON schema.
2. The model replies with a **tool call** instead of text: which tool, with which arguments. `finish_reason` is `"tool_calls"`.
3. *Your* code runs the function.
4. You send the result back as a `"tool"` message, and the model writes the final answer.

LangChain runs this loop for you in `d_langchain_agent`. One round by hand shows what its traces are recording.

</div>

In [9]:
def multiply(a: int, b: int) -> int:
    return a * b


# Step 1: describe the tool in the OpenAI tools format.
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What is 12 multiplied by 34? Use the tool."}]
response = client.chat.completions.create(
    model=TOOL_MODEL, messages=messages, tools=tools, reasoning_effort="none"
)

# Step 2: the model asks for a call instead of answering.
choice = response.choices[0]
print("finish_reason:", choice.finish_reason)
for call in choice.message.tool_calls or []:
    print("tool call:", call.function.name, call.function.arguments)

finish_reason: tool_calls
tool call: multiply {"a":12,"b":34}


In [10]:
if not choice.message.tool_calls:
    raise RuntimeError(
        "The model answered without calling the tool. Re-run the cell above."
    )

call = choice.message.tool_calls[0]
result = multiply(**json.loads(call.function.arguments))  # Step 3: *your* code runs it

follow_up = messages + [
    choice.message.model_dump(exclude_none=True),  # the model's tool-call turn
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": str(result),
    },  # Step 4: send the result back
]
final = client.chat.completions.create(
    model=TOOL_MODEL, messages=follow_up, tools=tools, reasoning_effort="none"
)
print("tool result: ", result)
print("final answer:", final.choices[0].message.content)

tool result:  408
final answer: The result of multiplying 12 by 34 is 408.


<div style="text-align: justify; max-width: 90ch">

Small models don't always make the call; that's the gap between `qwen3:0.6b` and `qwen3:1.7b` in the test above. When an agent misbehaves, check `finish_reason` and the tool-call arguments first.
In the GenAI track, MLflow traces record both for every step.

</div>

<div style="text-align: justify; max-width: 90ch">

## Troubleshooting

Find your symptom. Each entry names the cause, then the fix.

### `APIConnectionError` in Python, or `curl` can't connect

**Cause:** nothing is listening on port 11434, so the server isn't running. **Fix:** start it (table in Step 2), then re-run the cell.

### `Error: listen tcp 127.0.0.1:11434: bind: address already in use`

**Cause:** you ran `ollama serve`, but something already holds the port. Almost always that is Ollama itself, running as the service or app.
**Fix:** usually nothing. You don't need `ollama serve`, because a server is already running; check with `curl http://localhost:11434`.

- **Want a server in the terminal anyway** (e.g. to try a setting)? Stop the other one first: `sudo systemctl stop ollama` on Linux, or quit the app on macOS and Windows.
- **Another program holds the port?** Find it with `sudo lsof -i :11434` (Linux, macOS) or `netstat -ano | findstr 11434` (Windows).
  Or run Ollama on another port with `OLLAMA_HOST=127.0.0.1:11435 ollama serve`, and use `base_url="http://localhost:11435/v1"` in Python.

### `NotFoundError: Error code: 404` ... `model 'qwen3:4b' not found`

**Cause:** the server has no model with that exact name. **Fix:**

- Run `ollama list` and copy the name exactly. `qwen3` on its own means `qwen3:latest`, the 8b model, not `qwen3:1.7b`.
- Missing? `ollama pull <name>`.
- **Pulled it, but it vanished?** On Linux, if you stopped the service and started `ollama serve` yourself, that server reads models from `~/.ollama/models`,
  not from the service's folder. Go back to the service: stop your `ollama serve` and run `sudo systemctl start ollama`.

### Out of memory: the model won't load, an error mentions memory, or everything crawls

**Cause:** the model plus its context window doesn't fit in GPU memory or RAM. **Fix,** cheapest first:

- Unload models you aren't using: `ollama ps`, then `ollama stop <name>`.
- Close other memory-hungry programs, such as a notebook training a model or another model server.
- Lower the context window if you raised `OLLAMA_CONTEXT_LENGTH`.
- Use a smaller model: see the sizing table. On very small machines `gemma3:1b` loads where `gemma3:4b` won't, at the cost of answer quality;
  for `d_langchain_agent`, `qwen3:0.6b` loads where `qwen3:1.7b` won't, at the cost of reliable tool calls.

A CPU/GPU split in `ollama ps` is the early warning: the model loads, but only by spilling onto the CPU.

### The first call is slow or times out, and later calls are fast

**Cause:** the first call waits while Ollama loads the model from disk into memory. On a slow disk or a CPU-only machine that takes noticeably longer.
Ollama unloads an idle model after 5 minutes, so after a break you pay the loading time again. **Fix:**

- Expect it, and keep a generous client `timeout`, as the `client` cell above does.
- Warm the model up before a session: `ollama run gemma3:4b --verbose "hi"` prints the `load duration`.
- Keep it loaded longer: `ollama run gemma3:4b --keepalive 30m`, or set `OLLAMA_KEEP_ALIVE` on the server.
- If *every* call is slow, check that `ollama ps` shows the GPU you expect and, for a `qwen3` model, that thinking is off (Step 5).

### Empty answer with `finish_reason="length"`

**Cause:** a reasoning model such as `qwen3` spent the whole `max_tokens` budget thinking and never reached the answer. **Fix:** pass `reasoning_effort="none"`, or raise `max_tokens`.

### `ollama ps` shows `100% CPU` although you have a GPU

**Cause:** Ollama didn't find a usable GPU. **Fix:** update the GPU driver (NVIDIA: 550 or newer), restart Ollama, and read the server log for GPU errors:

- **Linux:** `journalctl -u ollama --no-pager --follow --pager-end`
- **macOS:** `cat ~/.ollama/logs/server.log`
- **Windows:** `server.log` in `%LOCALAPPDATA%\Ollama`

On Linux with an NVIDIA GPU, if the log shows initialization errors, Ollama's docs suggest reloading the driver module with `sudo rmmod nvidia_uvm && sudo modprobe nvidia_uvm`,
or rebooting.

### A setting you changed has no effect

**Cause:** the variable reached your terminal, not the server. **Fix:** set it where the server reads it and restart Ollama (settings section, "The gotcha").

### A request option such as `reasoning_effort` seems ignored

**Cause:** your Ollama may be older than the feature. **Fix:** update. On Linux, re-run the install script. On macOS and Windows, click the Ollama icon and choose **Restart to update** when it's offered.

</div>

<div style="text-align: justify; max-width: 90ch">

## Where to go next

You now have a free, private LLM behind an OpenAI-compatible URL. Continue with:

- **[`gen_ai/a_tracing_quickstart`](../gen_ai/a_tracing_quickstart.ipynb)**: trace your first LLM call with `mlflow.openai.autolog()`, using the same `base_url` and `api_key` as Step 4.
  It also needs the MLflow tracking server from [`basics/a_setup_mlflow`](../basics/a_setup_mlflow.ipynb).

Keep handy:

- **[Ollama documentation](https://docs.ollama.com):** the FAQ covers every server setting.
- **[Ollama model library](https://ollama.com/library):** other models and their tags. Before switching, run `ollama show <model>` and check its capabilities:
  the track's default needs only `completion`; `d_langchain_agent` needs `tools`; `i_rag_capstone` needs an `embedding` model.
- **Uninstalling:** the [Linux](https://docs.ollama.com/linux) and [macOS](https://docs.ollama.com/macos) pages end with removal commands.
  On Windows, use **Add or remove programs**; see the [Windows](https://docs.ollama.com/windows) page.

</div>